# Phase 2 · Lesson 2 — Fine-tuning

**Mastering Agentic AI Certification · Pre-read**

> Pre-training gives a model broad knowledge but no sense of *how to behave*. A base model will happily continue your text instead of answering your question. **Fine-tuning** is the stage that turns a broadly-knowledgeable **base model** into a focused, instruction-following model — without re-learning language from scratch.

## 1. What is fine-tuning?

**Fine-tuning** continues training a *pre-trained* model on a smaller, **curated** dataset so it specialises in a desired behaviour, format, or domain.

- It **reuses** everything learned during pre-training — grammar, facts, reasoning.
- It only **nudges** the existing weights toward a narrower goal.
- It is **orders of magnitude cheaper** than pre-training (hours/days on a few GPUs, not months on thousands).

The most important variant for agents is **Supervised Fine-Tuning (SFT)**, also called **instruction tuning**: the model is shown thousands of *(instruction → ideal response)* examples so it learns to *answer* rather than merely *continue*.

## 2. Where fine-tuning sits in the LLM lifecycle

```
  [1] PRE-TRAINING            -> base model: learns language + knowledge (self-supervised, huge data)
        |
  [2] FINE-TUNING / SFT       -> instruction-following on curated examples   <-- YOU ARE HERE
        |
  [3] ALIGNMENT (RLHF / DPO)  -> helpful, honest, harmless behaviour
        |
  [4] AGENTIC USE             -> tools, memory, planning, multi-step action
```

Fine-tuning is step **[2]**. It does **not** teach the model most of its knowledge — it *steers* the knowledge already inside the base model toward useful behaviour.

## 3. The core objective: same loss, curated data

SFT uses the **exact same next-token prediction objective** as pre-training — minimising cross-entropy on the next token:

$$\mathcal{L} = -\sum_{t} \log P_\theta(x_t \mid x_1, \dots, x_{t-1})$$

The difference is **what** the tokens are. Instead of raw web text, each example is a structured *(prompt, response)* pair, and the loss is typically computed **only on the response tokens**:

```
<|user|> Summarise photosynthesis in one sentence. <|assistant|> Plants convert sunlight, water, and CO2 into glucose and oxygen.
                                                    \____________________ loss is measured here ____________________/
```

> **Connection to Lesson 1's "label":** here the *response* tokens written by a human (or a stronger model) **are the labels**. So unlike self-supervised pre-training, SFT *is* supervised — but the volume of labelled data is tiny compared to the pre-training corpus.

## 4. Flavours of fine-tuning

| Method | What it changes | When to use |
|---|---|---|
| **Full fine-tuning** | Updates **all** weights | Maximum control; expensive; needs lots of memory |
| **PEFT / LoRA** | Freezes the base model; trains tiny **low-rank adapters** inserted into attention layers | Cheap, fast, swappable; the modern default |
| **SFT (instruction tuning)** | Teaches *(instruction → answer)* behaviour | Turning a base model into an assistant |
| **Domain fine-tuning** | Adapts to legal/medical/code/etc. text & style | Specialised vocabulary or formats |

**LoRA** is the key idea behind cheap fine-tuning: rather than editing the giant weight matrices, you learn a small `ΔW = B·A` (with `A`, `B` low-rank) and add it on. Far fewer parameters to train and store.

## 5. The ingredients of fine-tuning

| Ingredient | Pre-training | Fine-tuning |
|---|---|---|
| **Data** | Trillions of raw, unlabelled tokens | Thousands–millions of **curated, labelled** examples |
| **Objective** | Next-token prediction | Next-token prediction (often loss on response only) |
| **Compute** | Thousands of GPUs · weeks–months | A few GPUs · hours–days (less still with LoRA) |
| **Output** | Base / foundation model | Instruction-following / specialised model |

In [1]:
# Toy illustration of SFT: "steer" a base model with a few curated examples.
# We reuse Lesson 1's bigram idea. The BASE model just continues text;
# fine-tuning data teaches it to follow an "instruction" pattern.
from collections import defaultdict, Counter

# --- Pretend this is the pre-trained BASE model (broad but unfocused) ---
base_corpus = (
    "the model continues text the model predicts the next word "
    "the model continues text the model rambles on and on"
)

# --- Curated fine-tuning data: (instruction -> desired answer) demonstrations ---
# Note the special tokens that mark roles -- the model LEARNS this structure.
sft_corpus = (
    "<q> define ai <a> ai is software that performs tasks needing human intelligence <end> "
    "<q> define agent <a> an agent is an ai that plans and uses tools to act <end> "
    "<q> define agent <a> an agent is an ai that plans and uses tools to act <end>"
)

def train(corpus):
    t = defaultdict(Counter)
    toks = corpus.split()
    for a, b in zip(toks, toks[1:]):
        t[a][b] += 1
    return t

base = train(base_corpus)
# Fine-tuning = continue training the SAME model on curated data (here: merge counts)
finetuned = train(base_corpus + " " + sft_corpus)

def nextword(model, w):
    c = model[w]
    return c.most_common(3)

print("BASE      after '<a>':", nextword(base, "<a>"), "  <- never saw the instruction format")
print("FINETUNED after '<a>':", nextword(finetuned, "<a>"), " <- now answers in the taught format")
print("FINETUNED after '<q>':", nextword(finetuned, "<q>"))

BASE      after '<a>': []   <- never saw the instruction format
FINETUNED after '<a>': [('an', 2), ('ai', 1)]  <- now answers in the taught format
FINETUNED after '<q>': [('define', 3)]


In [ ]:
# Generate an 'answer' from the fine-tuned model once it sees the <a> cue.
def generate(model, start, n=8, stop="<end>"):
    out, w = [start], start
    for _ in range(n):
        nxt = model[w].most_common(1)
        if not nxt:
            break
        w = nxt[0][0]
        if w == stop:
            break
        out.append(w)
    return " ".join(out)

print("Prompted with '<a>':", generate(finetuned, "<a>"))

## 6. How fine-tuning contributes to the Transformer

Fine-tuning **does not change the Transformer architecture** — it adjusts the *parameters* of the same network the base model already uses. Concretely:

- The Transformer's layers (self-attention + feed-forward blocks) stay **identical**; only the **numbers inside the weight matrices** shift.
- **Full fine-tuning** updates every weight, including the attention projections `W_Q`, `W_K`, `W_V`, `W_O` and the MLP weights — the same matrices that pre-training produced.
- **LoRA / PEFT** literally **plugs into the Transformer's attention layers**: it freezes `W_Q`/`W_V` and learns a small additive `ΔW = B·A`. At inference the Transformer computes `(W + ΔW)x` — same forward pass, slightly nudged weights.
- Because the **self-attention mechanism is untouched**, the model keeps its long-context ability (vital for tool outputs, memory, and multi-turn agent loops) while gaining new behaviour.

> **In one line:** pre-training *builds* the Transformer's weights; fine-tuning *re-tunes a small slice of those same weights* so the same network behaves the way we want — no new architecture, no relearning language.

## 7. Why fine-tuning matters for agentic AI

- **Instruction-following** is what lets an agent reliably do what a plan/prompt asks.
- **Tool & function-call formatting** (emitting valid JSON, choosing tools) is taught largely via fine-tuning on tool-use traces.
- **Domain adaptation** (legal, medical, internal company data) lets an agent specialise without a new base model.
- **LoRA adapters are swappable** — one base model can host many task-specific agents cheaply.
- You usually fine-tune a **vendor base model via an API or PEFT**, not from scratch.

## 8. Key takeaways

1. **Fine-tuning** continues training a base model on **curated, labelled** data → an instruction-following / specialised model.
2. **SFT (instruction tuning)** uses the *same next-token loss*, but the human-written responses **are the labels** (it is supervised).
3. **LoRA / PEFT** make it cheap by training tiny low-rank adapters inside the **Transformer's attention layers** — the architecture is unchanged.
4. Fine-tuning **steers** the knowledge from pre-training; it does not replace it.
5. It is the stage that makes a model usable as the reasoning core of an **agent**.

---
*Next:* **alignment** (RLHF / DPO) — making the fine-tuned model helpful, honest, and harmless.

In [ ]:
# Environment sanity check — confirms the notebook is running in the dev container.
import sys, platform
print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())
print("Lesson 2 (Fine-tuning) notebook is running ✓")